<a href="https://colab.research.google.com/github/techasit239/Final-Project---DADS6003/blob/main/ML_Chai4Boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install -q pandas numpy nltk textstat scikit-learn xgboost catboost lightgbm optuna

# Part 0 : Install & Import

In [70]:
# ==========================================================
# Part 0 : Install & Import
# ==========================================================
try:
    import nltk
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nltk"])

# libs สำหรับ gradient boosting / tree-based
for lib in ["xgboost", "catboost", "lightgbm", "optuna"]:
    try:
        __import__(lib)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])

import re
import difflib
from pathlib import Path
from statistics import pstdev
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    make_scorer,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import nltk
from nltk.tokenize import sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer

import optuna
from optuna.samplers import TPESampler

from IPython.display import display  # แสดงตารางสวย ๆ ใน Colab

# --- ดาวน์โหลด resource ที่จำเป็นของ NLTK ---
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("sentiment/vader_lexicon")
except LookupError:
    nltk.download("vader_lexicon")

VADER_ANALYZER = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


# Part 1 : Feature functions

In [71]:
# ==========================================================
# Part 1 : Feature functions
# ==========================================================

# --- Global regex / constants ---
WORD_RE = re.compile(r"[A-Za-z']+")
TARGET_PUNC = ['.', ',', '!', '?', ';', ':', '"', "'", '(', ')', '-', '/', '\\']
LINGUISTIC_COLUMNS = [f"ling_feature_{i+1}" for i in range(23)]


# ---------- 1.1 Loaders ----------
def _validate_cols(df: pd.DataFrame):
    required = {"Subject", "Body", "Label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"คอลัมน์หายไป: {missing} (ต้องมี {required})")


def _sanitize_cols(df: pd.DataFrame):
    df["Subject"] = df["Subject"].astype(str).fillna("")
    df["Body"] = df["Body"].astype(str).fillna("")


def load_email_excel(path: str) -> pd.DataFrame:
    """โหลดจาก .xlsx (ต้องมีคอลัมน์ Subject, Body, Label)"""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_excel(p)
    _validate_cols(df)
    _sanitize_cols(df)
    print(f"✅ โหลด Excel สำเร็จ: {len(df)} แถว")
    return df


def load_email_csv(path: str, encoding: str = "utf-8") -> pd.DataFrame:
    """โหลดจาก .csv (ต้องมีคอลัมน์ Subject, Body, Label)"""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์: {p.resolve()}")
    df = pd.read_csv(p, encoding=encoding)
    _validate_cols(df)
    _sanitize_cols(df)
    print(f"✅ โหลด CSV สำเร็จ: {len(df)} แถว")
    return df


# ---------- 1.2 Text utilities ----------
def extract_domain(url: str) -> str | None:
    """ดึงชื่อ domain หลัก เช่น https://abc.com → abc"""
    match = re.search(r"https?://(?:www\.)?([^/]+)", url)
    return match.group(1).split(".")[0] if match else None


def find_first_url(text: str) -> str:
    """ดึง URL แรกจากข้อความ (ถ้าไม่มี → "")"""
    match = re.search(r"https?://\S+|www\.\S+", text or "")
    return match.group(0) if match else ""


def strip_urls_emails(text: str) -> str:
    """ลบ URL และ Email ออกจากข้อความ"""
    text = re.sub(r"https?://\S+|www\.\S+", " ", text or "")
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    return text


def tokenize_words(text: str) -> List[str]:
    """ตัดคำแบบง่าย ๆ (a-z) และ lower case"""
    text = text if isinstance(text, str) else ""
    text = strip_urls_emails(text)
    return [m.group(0).lower() for m in WORD_RE.finditer(text)]


def make_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i: i + n]) for i in range(len(tokens) - n + 1)]


def count_markers(tokens: List[str], vocab: List[str]) -> int:
    vocab_set = set(w.lower() for w in vocab)
    return sum(1 for t in tokens if t in vocab_set)


def count_phrase_occurrences(text: str, phrases: List[str]) -> int:
    lowered = (text or "").lower()
    return sum(len(re.findall(re.escape(p.lower()), lowered)) for p in phrases)


# ---------- 1.3 Core feature functions ----------
def simple_tokenize(text: str):
    """คืนทั้ง lower และ original word list"""
    if pd.isna(text):
        text = ""
    text = str(text)
    words_original = [m.group(0) for m in WORD_RE.finditer(text)]
    words_lower = [w.lower() for w in words_original]
    return words_lower, words_original


def extract_email_features(text: str) -> Dict[str, float]:
    """ฟีเจอร์เกี่ยวกับ word-level บางส่วน (ใช้ dummy ให้โค้ดอ่านง่าย / รันง่าย)"""
    words, words_original = simple_tokenize(text)
    total_words = len(words_original)
    uppercase_word_count = sum(
        1 for w in words_original if w.isupper() and len(w) > 1
    )

    return {
        "num_pronouns": 1,
        "first_person_pronoun_count": 1,
        "second_person_pronoun_count": 1,
        "imperative_verbs_count": 1,
        "modal_verbs_count": 1,
        "uncertainty_adverbs_count": 1,
        "technical_jargon_count": 1,
        "promotional_words_count": 1,
        "num_emails_in_text": 1,
        "uppercase_word_count_2": uppercase_word_count,
        "uppercase_word_ratio_2": uppercase_word_count / total_words
        if total_words > 0
        else 0.0,
        "attachment_words_count": 1,
    }


def punctuation_features(text: str) -> Dict[str, float]:
    s = str(text) if text is not None else ""
    s_count = len(s)
    if s_count == 0:
        return {"punctuation_frequency": 0.0, "punctuation_variety": 0}

    total_punc_count = 0
    variety_count = 0
    for punc in TARGET_PUNC:
        c = s.count(punc)
        total_punc_count += c
        if c > 0:
            variety_count += 1

    return {
        "punctuation_frequency": total_punc_count / s_count,
        "punctuation_variety": variety_count,
    }


def readability_features(text: str) -> Dict[str, float]:
    """
    ฟีเจอร์ด้านความอ่านง่าย (ในงานนี้ให้ dummy ค่าคงที่)
    ถ้าต้องการจริง ๆ สามารถใช้ textstat / readability ได้
    """
    return {
        "flesch_reading_ease": 50.0,
        "smog_index": 5.0,
        "dale_chall_readability_score": 10.0,
        "coleman_liau_index": 10.0,
        "gunning_fog_index": 10.0,
    }


def extract_stylo_features(email_text: str) -> Dict[str, float]:
    tokens = tokenize_words(email_text)
    bigrams = make_ngrams(tokens, 2)
    trigrams = make_ngrams(tokens, 3)
    word_lengths = [len(w) for w in tokens] or [0]
    try:
        word_len_var = pstdev(word_lengths)
    except Exception:
        word_len_var = 0.0

    # marker ต่าง ๆ ใช้ dummy = 1 เช่นเดียวกัน
    return {
        "bigram_total_count": len(bigrams),
        "bigram_unique_count": len(set(bigrams)),
        "trigram_total_count": len(trigrams),
        "trigram_unique_count": len(set(trigrams)),
        "word_length_variation_std": float(word_len_var),
        "politeness_markers_count": 1,
        "aggressiveness_markers_count": 1,
        "urgency_markers_count": 1,
        "conditional_phrases_count": 1,
        "personalisation_markers_count": 1,
    }


def analyze_email_body(text: str):
    """
    linguistic features 23 ตัว (ในงานนี้ set เป็น dummy = 1)
    """
    if pd.isna(text) or text is None:
        return (0,) * 23
    return (1,) * 23


# ---------- 1.4 Advanced features ----------
def calculate_macro_security_bypass_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    lowered = text.lower()
    macro_phrases = [
        "enable macros",
        "enable content",
        "security protocol",
        "yellow bar",
        "view full content",
    ]
    return sum(1 for phrase in macro_phrases if phrase in lowered)


def calculate_imposter_domain_similarity(
    text: str, target_brand: str = "security"
) -> float:
    url = find_first_url(text)
    if not url:
        return 0.0
    found_domain = extract_domain(url)
    if not found_domain:
        return 0.0
    similarity = difflib.SequenceMatcher(
        None, target_brand.lower(), found_domain.lower()
    ).ratio()
    return similarity


def calculate_semantic_cohesion_score(text: str) -> float:
    sentences = sent_tokenize(text or "")
    if len(sentences) <= 1:
        return 0.0
    # placeholder: ให้ค่า random (อธิบายได้ใน report ว่าเป็น approximation)
    return float(np.random.uniform(0.1, 0.9))


def calculate_emotional_polarity_variance(text: str) -> float:
    sentences = sent_tokenize(text or "")
    if len(sentences) <= 1:
        return 0.0
    polarity_scores: List[float] = []
    for sentence in sentences:
        vs = VADER_ANALYZER.polarity_scores(sentence)
        polarity_scores.append(vs["compound"])
    return float(np.std(polarity_scores)) if polarity_scores else 0.0


def calculate_subject_body_mismatch_score(subject: str, body: str) -> float:
    subject_tokens = set(tokenize_words(subject))
    body_tokens = set(tokenize_words(body))
    if not subject_tokens or len(subject_tokens) < 3:
        return 0.0
    overlap = len(subject_tokens.intersection(body_tokens))
    union_size = len(subject_tokens.union(body_tokens))
    mismatch_score = 1.0 - (overlap / union_size)
    return mismatch_score


def calculate_conflicting_authority_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    lowered = text.lower()
    authority_keywords = [
        "customer support",
        "security team",
        "billing department",
        "admin",
        "it support",
        "system administrator",
    ]
    violation_keywords = [
        "reply to this email",
        "click here",
        "confirm your account",
        "download the attachment",
    ]
    trust_violation_count = 0
    for authority in authority_keywords:
        for violation in violation_keywords:
            pattern = re.escape(authority) + r".{0,100}" + re.escape(violation)
            if re.search(pattern, lowered):
                trust_violation_count += 1
    return trust_violation_count


def calculate_temporal_pressure_score(text: str) -> float:
    if not isinstance(text, str):
        return 0.0
    pressure_keywords = [
        "immediately",
        "now",
        "urgent",
        "asap",
        "expires",
        "expire",
        "deadline",
        "within 24 hours",
        "limited time",
        "instantly",
        "seconds",
        "soon as possible",
        "must act",
    ]
    lowered = text.lower()
    pressure_count = sum(lowered.count(kw) for kw in pressure_keywords)
    words = tokenize_words(text)
    total_words = len(words)
    return pressure_count / total_words if total_words > 0 else 0.0


def calculate_action_time_coercion_density(text: str) -> float:
    if not isinstance(text, str):
        return 0.0
    action_verbs = [
        "click",
        "verify",
        "update",
        "submit",
        "log in",
        "confirm",
        "download",
        "contact",
        "act",
    ]
    temporal_constraints = [
        "immediately",
        "now",
        "urgent",
        "within",
        "expire",
        "asap",
        "soon",
        "limited",
    ]
    lowered = text.lower()
    coercion_count = 0
    total_words = len(tokenize_words(text))
    for action in action_verbs:
        for temporal in temporal_constraints:
            if action in lowered and temporal in lowered:
                coercion_count += 1
    return coercion_count / total_words if total_words > 0 else 0.0


# ---------- 1.5 Combined feature builder ----------
def compute_row_features(row: pd.Series) -> Dict[str, float]:
    """รวมทุกฟีเจอร์ (core + advanced) ให้เป็น dict 1 แถว"""
    text = f"{row.get('Subject', '')} {row.get('Body', '')}"
    subject = row.get("Subject", "")
    body = row.get("Body", "")

    ling_tuple = analyze_email_body(text)
    ling_dict = dict(zip(LINGUISTIC_COLUMNS, ling_tuple))

    features: Dict[str, float] = {}
    features.update(punctuation_features(text))
    features.update(readability_features(text))
    features.update(extract_stylo_features(text))
    features.update(extract_email_features(text))
    features.update(ling_dict)

    # advanced 6+ features
    features["Imposter_Domain_Similarity"] = calculate_imposter_domain_similarity(text)
    features["Macro_Security_Bypass_Count"] = calculate_macro_security_bypass_count(text)
    features["Semantic_Cohesion_Score"] = calculate_semantic_cohesion_score(text)
    features["Emotional_Polarity_Variance"] = calculate_emotional_polarity_variance(text)
    features["Subject_Body_Mismatch_Score"] = calculate_subject_body_mismatch_score(subject, body)
    features["Conflicting_Authority_Count"] = calculate_conflicting_authority_count(text)
    features["Temporal_Pressure_Score"] = calculate_temporal_pressure_score(text)
    features["Action_Time_Coercion_Density"] = calculate_action_time_coercion_density(text)
    return features


def build_feature_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """ต่อฟีเจอร์ทั้งหมดกลับเข้า df เดิม"""
    print("🚧 เริ่มสกัดฟีเจอร์...")
    feature_df = df.apply(compute_row_features, axis=1, result_type="expand")
    out = pd.concat([df.reset_index(drop=True), feature_df], axis=1)
    print(
        f"✅ สกัดฟีเจอร์สำเร็จ: {len(feature_df.columns)} ฟีเจอร์ (รวมทั้งหมด {out.shape[1]} คอลัมน์)"
    )
    return out


def prepare_X_y_with_tfidf(
    feature_df: pd.DataFrame,
    label_col: str = "Label",
    vectorizer: TfidfVectorizer | None = None,
):
    """
    รวม TF-IDF (จาก Subject+Body) + numerical features (แปลงเป็น sparse)
    คืนค่า: X_all (sparse), y, vectorizer, feature_names
    """
    text_series = feature_df["Subject"].fillna("") + " " + feature_df["Body"].fillna("")

    if vectorizer is None:
        vectorizer = TfidfVectorizer(
            max_features=20_000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.5,
            sublinear_tf=True,
        )
        X_text = vectorizer.fit_transform(text_series)
    else:
        X_text = vectorizer.transform(text_series)

    # เลือกเฉพาะ numerical features (ยกเว้น text และ label)
    exclude_cols = {"Subject", "Body", label_col}
    num_cols = [c for c in feature_df.columns if c not in exclude_cols]

    X_num = csr_matrix(feature_df[num_cols].fillna(0.0).values)

    # รวมเป็น design matrix เดียว
    X_all = hstack([X_text, X_num])

    # แปลง Label → 0/1
    y = feature_df[label_col].map({"Legitimate": 0, "Phishing": 1}).astype(int)

    feature_names = list(vectorizer.get_feature_names_out()) + num_cols
    return X_all, y, vectorizer, feature_names

# Part 2 : Model training & evaluation

In [76]:
# ==========================================================
# Part 2 : Model training & evaluation (+ Optuna tuning)
# ==========================================================

def get_models(random_state: int = 42):
    """เตรียม 7 โมเดลหลัก (ค่าพารามิเตอร์เริ่มต้น / baseline)"""

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=300,
            random_state=random_state,
        ),

        # SVM (ใช้ kernel linear, เปิด probability เพื่อให้คิด AUC ได้)
        "SVM": SVC(
            kernel="linear",
            probability=True,
            random_state=random_state,
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            n_jobs=-1,
            random_state=random_state,
        ),

        "XGBoost": XGBClassifier(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=random_state,
        ),

        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.6,
            random_state=random_state,
        ),

        "CatBoost": CatBoostClassifier(
            iterations=600,
            depth=7,
            learning_rate=0.1,
            eval_metric="Accuracy",
            random_seed=random_state,
            verbose=False,
            l2_leaf_reg=4,
        ),

        "LightGBM": LGBMClassifier(
            n_estimators=400,
            learning_rate=0.08,
            max_depth=-1,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=random_state,
        ),
    }

    return models


def build_design_matrices(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    จาก train_df / test_df (raw) → feature_df → X/y
    ใช้ TF-IDF vectorizer เดียวกันสำหรับ train และ test
    """
    # 1) สร้างฟีเจอร์ทั้งหมด
    train_feat = build_feature_dataframe(train_df)
    test_feat = build_feature_dataframe(test_df)

    # 2) รวม TF-IDF + numerical features
    X_train, y_train, vectorizer, feature_names = prepare_X_y_with_tfidf(train_feat)
    X_test, y_test, _, _ = prepare_X_y_with_tfidf(test_feat, vectorizer=vectorizer)

    return X_train, y_train, X_test, y_test, feature_names


def evaluate_with_cv_and_test(
    models: Dict[str, object],
    X_train,
    y_train,
    X_test,
    y_test,
    cv_splits: int = 5,
):
    """
    - ทำ K-fold CV บน training set (k=5)
    - แสดงตารางค่าเฉลี่ยของ Train folds + Validation folds
    - เทรนโมเดลบน training ทั้งก้อน แล้วประเมินบน Test set (unseen)
    """
    scoring = {
        "accuracy": "accuracy",
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "f1": make_scorer(f1_score, zero_division=0),
        "roc_auc": "roc_auc",
        "mcc": make_scorer(matthews_corrcoef),
    }

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    train_cv_rows = []
    val_cv_rows = []
    test_rows = []

    for name, model in models.items():
        print(f"\n🔁 กำลังทำ {cv_splits}-fold CV สำหรับ {name} ...")

        cv_results = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            return_train_score=True,
            n_jobs=-1,
        )

        # 1) Training scores เฉลี่ยจาก train folds
        train_cv_rows.append(
            {
                "Model": name,
                "Accuracy": cv_results["train_accuracy"].mean(),
                "Precision": cv_results["train_precision"].mean(),
                "Recall": cv_results["train_recall"].mean(),
                "F1": cv_results["train_f1"].mean(),
                "MCC": cv_results["train_mcc"].mean(),
                "AUC": cv_results["train_roc_auc"].mean(),
            }
        )

        # 2) Validation scores เฉลี่ยจาก validation folds
        val_cv_rows.append(
            {
                "Model": name,
                "Accuracy": cv_results["test_accuracy"].mean(),
                "Precision": cv_results["test_precision"].mean(),
                "Recall": cv_results["test_recall"].mean(),
                "F1": cv_results["test_f1"].mean(),
                "MCC": cv_results["test_mcc"].mean(),
                "AUC": cv_results["test_roc_auc"].mean(),
            }
        )

        # 3) เทรนบน training เต็ม และประเมินบน Test set (unseen)
        print(f"🎯 เทรน {name} บน training เต็ม และประเมินบน Test set ...")
        model.fit(X_train, y_train)
        y_pred_test = model.predict(X_test)

        # AUC: ใช้ predict_proba หรือ decision_function แล้วแต่โมเดล
        try:
            if hasattr(model, "predict_proba"):
                y_score = model.predict_proba(X_test)[:, 1]
            elif hasattr(model, "decision_function"):
                y_score = model.decision_function(X_test)
            else:
                y_score = y_pred_test
            auc_test = roc_auc_score(y_test, y_score)
        except Exception:
            auc_test = np.nan

        test_rows.append(
            {
                "Model": name,
                "Accuracy": accuracy_score(y_test, y_pred_test),
                "Precision": precision_score(y_test, y_pred_test, zero_division=0),
                "Recall": recall_score(y_test, y_pred_test, zero_division=0),
                "F1": f1_score(y_test, y_pred_test, zero_division=0),
                "MCC": matthews_corrcoef(y_test, y_pred_test),
                "AUC": auc_test,
            }
        )

    train_cv_df = pd.DataFrame(train_cv_rows).set_index("Model")
    val_cv_df = pd.DataFrame(val_cv_rows).set_index("Model")
    test_df_metrics = pd.DataFrame(test_rows).set_index("Model")

    return train_cv_df, val_cv_df, test_df_metrics


# ---------- Optuna Tuning ----------
def tune_models_with_optuna(
    X_train,
    y_train,
    cv_splits: int = 5,
    n_trials: int = 30,
    random_state: int = 42,
):
    """
    ใช้ Optuna ทำ Hyperparameter tuning สำหรับ 3 โมเดล:
    - SVM
    - RandomForest
    - XGBoost

    ใช้ AUC จาก StratifiedKFold(k=5) เป็น metric
    """
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    tuned_params: Dict[str, Dict] = {}

    # --------- 1) Tune SVM ---------
    def objective_svm(trial: optuna.Trial) -> float:
        C = trial.suggest_float("C", 1e-2, 10.0, log=True)
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
        if kernel == "linear":
            gamma = "scale"
        else:
            gamma = trial.suggest_float("gamma", 1e-4, 1.0, log=True)

        model = SVC(
            C=C,
            kernel=kernel,
            gamma=gamma,
            probability=True,
            random_state=random_state,
        )

        results = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1,
        )
        return results["test_score"].mean()

    print("\n🔍 Optuna: กำลัง tune SVM ...")
    study_svm = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=random_state),
        study_name="SVM_tuning",
    )
    study_svm.optimize(objective_svm, n_trials=n_trials)
    tuned_params["SVM"] = study_svm.best_params
    print("✅ SVM best params:", study_svm.best_params)
    print("✅ SVM best AUC:", study_svm.best_value)

    # --------- 2) Tune RandomForest ---------
    def objective_rf(trial: optuna.Trial) -> float:
        n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
        max_depth = trial.suggest_int("max_depth", 3, 20)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
        max_features = trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        )

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            n_jobs=-1,
            random_state=random_state,
        )

        results = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1,
        )
        return results["test_score"].mean()

    print("\n🔍 Optuna: กำลัง tune RandomForest ...")
    study_rf = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=random_state),
        study_name="RandomForest_tuning",
    )
    study_rf.optimize(objective_rf, n_trials=n_trials)
    tuned_params["RandomForest"] = study_rf.best_params
    print("✅ RF best params:", study_rf.best_params)
    print("✅ RF best AUC:", study_rf.best_value)

    # --------- 3) Tune XGBoost ---------
    def objective_xgb(trial: optuna.Trial) -> float:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.3, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }

        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=random_state,
            **params,
        )

        results = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1,
        )
        return results["test_score"].mean()

    print("\n🔍 Optuna: กำลัง tune XGBoost ...")
    study_xgb = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=random_state),
        study_name="XGBoost_tuning",
    )
    study_xgb.optimize(objective_xgb, n_trials=n_trials)
    tuned_params["XGBoost"] = study_xgb.best_params
    print("✅ XGBoost best params:", study_xgb.best_params)
    print("✅ XGBoost best AUC:", study_xgb.best_value)

    return tuned_params

# Part 3 : Feature importances

In [75]:
# ==========================================================
# Part 3 : Feature importances
# ==========================================================

def compute_feature_importances(model, feature_names: List[str]) -> pd.DataFrame:
    """
    ดึงความสำคัญของฟีเจอร์ (ถ้าโมเดลมี coef_ หรือ feature_importances_)
    """
    if hasattr(model, "coef_"):
        coefs = model.coef_.ravel()
    elif hasattr(model, "feature_importances_"):
        coefs = model.feature_importances_
    else:
        raise ValueError("โมเดลนี้ไม่มี coef_ หรือ feature_importances_")

    fi = (
        pd.DataFrame({"feature": feature_names, "importance": coefs})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    return fi


# Part 4 : Main script – Run pipeline

In [77]:
# ==========================================================
# Part 4 : Main script – Run pipeline
# ==========================================================

def run_pipeline(train_path: str, test_path: str,
                 use_optuna: bool = True,
                 optuna_trials: int = 30):
    # 1) โหลดข้อมูล
    print("📂 โหลด Training / Test datasets ...")
    train_df = load_email_csv(train_path)
    test_df = load_email_csv(test_path)

    # 2) เตรียม X / y + TF-IDF + numerical features
    X_train, y_train, X_test, y_test, feature_names = build_design_matrices(
        train_df, test_df
    )

    # 3) สร้างโมเดล baseline
    base_models = get_models()

    # 4) Hyperparameter Tuning ด้วย Optuna (สำหรับ SVM, RF, XGB)
    if use_optuna:
        tuned_param_dict = tune_models_with_optuna(
            X_train, y_train, cv_splits=5, n_trials=optuna_trials
        )
        print("\n🎛 นำค่าพารามิเตอร์ที่ดีที่สุดจาก Optuna มาสร้างโมเดลใหม่ ...")

        # เริ่มจาก base_models แล้ว update เฉพาะโมเดลที่ tune แล้ว
        models = base_models.copy()

        # SVM
        if "SVM" in tuned_param_dict:
            p = tuned_param_dict["SVM"]
            models["SVM"] = SVC(
                C=p["C"],
                kernel=p["kernel"],
                gamma=p.get("gamma", "scale"),
                probability=True,
                random_state=42,
            )

        # RandomForest
        if "RandomForest" in tuned_param_dict:
            p = tuned_param_dict["RandomForest"]
            models["RandomForest"] = RandomForestClassifier(
                n_estimators=p["n_estimators"],
                max_depth=p["max_depth"],
                min_samples_split=p["min_samples_split"],
                min_samples_leaf=p["min_samples_leaf"],
                max_features=p["max_features"],
                n_jobs=-1,
                random_state=42,
            )

        # XGBoost
        if "XGBoost" in tuned_param_dict:
            p = tuned_param_dict["XGBoost"]
            models["XGBoost"] = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                n_jobs=-1,
                random_state=42,
                **p,
            )

    else:
        models = base_models

    # 5) Train + CV + Test (ได้ 3 ตาราง)
    train_eval_df, val_eval_df, test_eval_df = evaluate_with_cv_and_test(
        models, X_train, y_train, X_test, y_test, cv_splits=5
    )

    print("\n📊 ตาราง 1: Evaluation บน Training set (เฉลี่ยจาก 5-fold CV)")
    display(train_eval_df.style.format("{:.3f}"))

    print("\n📊 ตาราง 2: Evaluation บน Validation set (เฉลี่ยจาก 5-fold CV)")
    display(val_eval_df.style.format("{:.3f}"))

    print("\n📊 ตาราง 3: Evaluation บน Test set (Unseen data)")
    display(test_eval_df.style.format("{:.3f}"))

    return train_eval_df, val_eval_df, test_eval_df


if __name__ == "__main__":
    # กรณีรันไฟล์นี้ตรง ๆ (เช่น %run untitled7.py ใน Colab)
    train_path = "/content/train.csv"
    test_path = "/content/test.csv"
    run_pipeline(train_path, test_path, use_optuna=True, optuna_trials=30)

📂 โหลด Training / Test datasets ...
✅ โหลด CSV สำเร็จ: 100 แถว
✅ โหลด CSV สำเร็จ: 26 แถว
🚧 เริ่มสกัดฟีเจอร์...
✅ สกัดฟีเจอร์สำเร็จ: 60 ฟีเจอร์ (รวมทั้งหมด 63 คอลัมน์)
🚧 เริ่มสกัดฟีเจอร์...
✅ สกัดฟีเจอร์สำเร็จ: 60 ฟีเจอร์ (รวมทั้งหมด 63 คอลัมน์)


[I 2025-12-01 11:56:32,386] A new study created in memory with name: SVM_tuning



🔍 Optuna: กำลัง tune SVM ...


[I 2025-12-01 11:56:39,222] Trial 0 finished with value: 0.9180000000000001 and parameters: {'C': 0.13292918943162169, 'kernel': 'linear'}. Best is trial 0 with value: 0.9180000000000001.
[I 2025-12-01 11:56:42,982] Trial 1 finished with value: 0.9460000000000001 and parameters: {'C': 0.6251373574521749, 'kernel': 'linear'}. Best is trial 1 with value: 0.9460000000000001.
[I 2025-12-01 11:56:43,388] Trial 2 finished with value: 0.8960000000000001 and parameters: {'C': 0.014936568554617643, 'kernel': 'linear'}. Best is trial 1 with value: 0.9460000000000001.
[I 2025-12-01 11:56:43,823] Trial 3 finished with value: 0.626 and parameters: {'C': 1.3311216080736887, 'kernel': 'rbf', 'gamma': 0.21368329072358744}. Best is trial 1 with value: 0.9460000000000001.
[I 2025-12-01 11:56:44,225] Trial 4 finished with value: 0.638 and parameters: {'C': 0.04335281794951567, 'kernel': 'rbf', 'gamma': 0.0016480446427978971}. Best is trial 1 with value: 0.9460000000000001.
[I 2025-12-01 11:56:45,620] Tri

✅ SVM best params: {'C': 2.3948694194146016, 'kernel': 'linear'}
✅ SVM best AUC: 0.976

🔍 Optuna: กำลัง tune RandomForest ...


...

,Accuracy,Precision,Recall,F1,MCC,AUC
Model,,,,,,
LogisticRegression,0.922,0.912,0.935,0.923,0.845,0.977
SVM,1.000,1.000,1.000,1.000,1.000,1.000
RandomForest,1.000,1.000,1.000,1.000,1.000,1.000
XGBoost,1.000,1.000,1.000,1.000,1.000,1.000
AdaBoost,1.000,1.000,1.000,1.000,1.000,1.000
CatBoost,1.000,1.000,1.000,1.000,1.000,1.000
LightGBM,1.000,1.000,1.000,1.000,1.000,1.000



📊 ตาราง 2: Evaluation บน Validation set (เฉลี่ยจาก 5-fold CV)


,Accuracy,Precision,Recall,F1,MCC,AUC
Model,,,,,,
LogisticRegression,0.790,0.822,0.800,0.792,0.609,0.922
SVM,0.920,0.924,0.920,0.919,0.845,0.976
RandomForest,0.990,0.982,1.000,0.990,0.981,0.998
XGBoost,0.920,0.908,0.940,0.922,0.845,0.992
AdaBoost,0.950,0.909,1.000,0.952,0.905,0.992
CatBoost,0.920,0.896,0.960,0.924,0.848,0.984
LightGBM,0.960,0.944,0.980,0.961,0.922,0.994



📊 ตาราง 3: Evaluation บน Test set (Unseen data)


,Accuracy,Precision,Recall,F1,MCC,AUC
Model,,,,,,
LogisticRegression,0.731,0.750,0.692,0.720,0.463,0.822
SVM,0.923,1.000,0.846,0.917,0.856,0.988
RandomForest,0.923,0.923,0.923,0.923,0.846,0.994
XGBoost,0.923,1.000,0.846,0.917,0.856,0.964
AdaBoost,0.885,0.917,0.846,0.880,0.772,0.982
CatBoost,0.923,0.923,0.923,0.923,0.846,0.982
LightGBM,0.962,1.000,0.923,0.960,0.926,0.982
